In [277]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go
import trimesh

from mano_pybullet.hand_model import HandModel20

In [278]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel, rotation_matrix_from_vectors
from model.hand_opt import AdamGraspTransfer

In [279]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

def rvec2mat(rvec):
    """Convert rotation vector to rotation matrix."""
    angle = np.linalg.norm(rvec)
    axis = rvec if angle != 0.0 else [0.0, 0.0, 1.0]
    mat = axangle2mat(axis, angle)
    return mat

In [280]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

env: MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models


## Set Sample Data

In [281]:
# Load data for the 00100 frame
# fname = "sample_hamer_output.npz"

frame_id = "000033"
# frame_id = "000172"
# frame_id = "000247"
# frame_id = "000072"

fname = f"{frame_id}.npz"


data = np.load(f"../data/{fname}", allow_pickle=True)

In [282]:
for k in data.keys():
  print(k)

pred_cam
pred_mano_params
pred_cam_t
focal_length
pred_keypoints_3d
pred_vertices
pred_keypoints_2d
opt_translation
bboxes
right
target_transfer_pose


## Set Left/Right

In [283]:
use_left_hand = True
print("Use left hand? -->", use_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print(left_idxs)
print(right_idxs)

idx_to_use = left_idxs if use_left_hand else right_idxs
print(use_left_hand, idx_to_use)
print(left_idxs.size, right_idxs.size)

Use left hand? --> True
[0]
[]
True [0]
1 0


In [284]:
mano_params = data['pred_mano_params'].item()
print(type(mano_params))
print(mano_params.keys())
print(mano_params['hand_pose'].shape) # for 2 hands

<class 'dict'>
dict_keys(['global_orient', 'hand_pose', 'betas'])
(1, 15, 3, 3)


In [285]:
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0].copy()
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0].copy()
mano_trans = data['opt_translation'][idx_to_use][0].copy()
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)
print(mano_trans.shape)

(3, 3)
(15, 3, 3)
(3,)


In [286]:
# See: https://github.com/geopavlakos/hamer/issues/61#issuecomment-2304863248
# See: https://github.com/geopavlakos/hamer/blob/dc19e5686198a7c3fc3938bff3951f238a85fd11/hamer/datasets/utils.py#L378
if use_left_hand:
  hand_rotn_mat[1::3] *= -1
  hand_rotn_mat[2::3] *= -1
  hand_theta_mat[1::3] *= -1
  hand_theta_mat[1::3] *= -1
  pass

In [287]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape, '\n', hand_theta_full)

(16, 3) 
 [[ 0.3324524   0.93764865  2.20776553]
 [ 0.06226856 -0.45119326  0.60342343]
 [ 0.22986924  0.0519553   0.32660566]
 [ 0.26371567 -0.19974491  0.3061791 ]
 [-0.01831745 -0.23496812  0.84491287]
 [-0.24735273 -0.11917977  0.41447235]
 [-0.05822115 -0.22250468  0.14586782]
 [ 0.14345386  0.8331534   0.3938154 ]
 [-0.4309592  -0.31133691  0.47567021]
 [-0.29450587 -0.14793427  0.36078861]
 [ 0.13797808  0.12426751  0.65082485]
 [-0.44392436 -0.14066906  0.51859464]
 [-0.08965547 -0.29806403  0.32141107]
 [ 0.96755534 -0.16220519 -0.13849951]
 [-0.68415387  0.05197677  0.20522206]
 [ 0.22917128  0.11407762  0.09980606]]


## Init Gripper Models

In [288]:
source_gripper = "mano_left" if use_left_hand else "mano_right"
# source_gripper = "mano_right"

target_gripper = "fetch_gripper"
device = "cpu"
print(device, source_gripper, target_gripper)

cpu mano_left fetch_gripper


In [289]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

# sm_left = get_handmodel(
#   "mano_left",
#   1,
#   device,
#   json_path="urdf_assets_meta.json",
#   datadir="../grippers/"
# )


In [290]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Grasp Pose

In [291]:
# Mano Pybullet Model

hand_model = HandModel20(left_hand=use_left_hand)
# hand_model = HandModel20(left_hand=False)

angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)

# hm_left = HandModel20(left_hand=True)
# angles_left, palm_basis_left = hm_left.mano_to_angles(hand_theta_full)


# print(len(angles))
# print(palm_basis)

# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155
origin = hand_model.origins()[0]
# origin = hm_left.origins()[0]

palm_trans = mano_trans + origin - palm_basis @ origin
print(palm_trans)
print(mano_trans)

actual_trans = np.array(palm_trans)
# if use_left_hand:
#   # actual_trans = palm_trans
#   actual_trans -= mano_trans
#   actual_trans[0] *= -1
#   actual_trans += mano_trans

# print(palm_trans)
print(actual_trans)
# palm_trans = np.array([palm_trans[0], mano_trans[1], mano_trans[2]])

[0.0264577  0.05538054 1.10956516]
[ 0.19058454 -0.01728491  1.11571178]
[0.0264577  0.05538054 1.10956516]


In [292]:
# hand_theta_full[0] = np.zeros(3)

# print(hm_left.mano_to_angles(hand_theta_full)[1])

In [293]:
actual_basis = palm_basis
if use_left_hand:
  R_x = np.array([
      [1, 0, 0],
      [0, -1, 0],
      [0, 0, -1]
  ])
  actual_basis = np.dot(actual_basis, R_x)
  pass

# if use_left_hand:
#   print("Updating grasp pose rotn and posn")
#   r_palm_normal = palm_basis @ np.array([0, -1, 0])
#   r_palm_normal_flip = np.array(r_palm_normal)
#   r_palm_normal_flip[0] *= -1
#   rotmat_flip = rotation_matrix_from_vectors(r_palm_normal, r_palm_normal_flip)
#   actual_basis = rotmat_flip @ palm_basis  


In [294]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3:] = torch.tensor(actual_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(actual_trans)
print("Pose:", grasp_pose)

grasp_dofs = -1 * torch.tensor(angles) if use_left_hand else torch.tensor(angles)
# grasp_dofs = torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([ 0.0265,  0.0554,  1.1096, -0.7188,  0.6943, -0.0361,  0.5081,  0.4891,
        -0.7090])
DOFS: tensor([-0.5317,  0.5583,  0.3324,  0.2831, -0.2043,  0.8453,  0.3841,  0.1503,
         0.8030,  0.5194,  0.1758,  0.1315,  0.0883,  0.6672,  0.3792,  0.2794,
        -0.9614, -0.2889,  0.2384, -0.1512])


In [295]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

min_energy_index = energy.min(dim=0)[1]
print(min_energy_index.item())

print(q_traj.shape)
best_q = q_traj[min_energy_index.item(), -1]
print(best_q.shape)

0
torch.Size([32, 301, 9])
torch.Size([9])


In [296]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, (target_model.dynamic_joints_q_upper[0] - target_model.dynamic_joints_q_mid[0])), dim=0)

## Viz Src + Target

In [297]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
target_gripper_mesh_data = target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='orange', opacity=0.2)
vis_data += target_gripper_mesh_data
fig = go.Figure(data=vis_data)
fig.show()


Plotting TARGET and SOURCE together...


In [298]:
trimesh_list = []

for mesh in target_gripper_mesh_data:
    vertices = np.array([mesh.x, mesh.y, mesh.z]).T
    faces = np.array([mesh.i, mesh.j, mesh.k]).T
    trimesh_list.append(trimesh.Trimesh(vertices=vertices, faces=faces))

combined_mesh = trimesh.util.concatenate(trimesh_list)
combined_mesh.export(f'../data/target_mesh_{frame_id}_{int(not use_left_hand)}.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 987\nproperty float x\nproperty float y\nproperty float z\nelement face 1962\nproperty list uchar int vertex_indices\nend_header\nY3\x03>\x04\x04\xaa<\n\x1d\x83?Z\x1a\x02>\x08\xa5\xa5<~\xfe\x82?M\x07\xf4=\xb7\x95\x15=\x05\xf3\x82?>\x08\xdc=\xeb6=<\xb0\x0c\x81?2J\xdc=\xd0\xcf:<\xfb\r\x81?#\xc2\xb9=\xf07\xf5<\xf2_\x80?N\xc3\xa1=\xc5G\xf3=d\xc1\x82?\xbe\xff\xa0=7\\\xf4=[\xc5\x82?$\x97\xa5=\xa7\x12\xf2=h\xd7\x82?N\x07P=\x92\xf8\xe5=nY\x82?\x8c(\x81<JW\xc9=\xfe\x8d\x8d?\x02\xea\x8a=@\xc9\xf9=\xa5(\x84?FR,\xbaC\xfe\xb5=y\x89\x8b?\x83pL=\x9d\xb9\xe4=\xdd>\x82?\xfd\x1e\x8b=]\xc8\x84=\x1aP\x80?)\xc2\x81=\xe3h\xa7=\x10D\x80?\xeb#\x8e=\xea\x19\x86=\xc4\'\x80?j"\x00>\xf9\xe9\xb9=\x1d[\x86?\x84y\x03>vn\xa9=\'\x93\x86?24\x04>\x01\x13\xaa=-f\x86?G\xf9\x07>(\xed\x99==h\x86?2\xb9\xf5=\xc8s\xc7=^\xba\x85?5\xa8\xd4=Ba\xfe=\x01Y\x85?S\x16\xd6=a\xc6\xff=\x1b~\x85?\xbba\xca=uM\x03>\xf0#\x87?1\xce\xc8=C&\x03>\x0

In [299]:
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)


# samp = sample_grasp_q.clone()
# samp[0, :9] = 0
# samp[0, 3] = 1
# samp[0, 7] = 1

# spleft = q_left.clone()
# spleft[0, :9] = 0
# spleft[0, 3] = 1
# # spleft[0, 7] = 1

# vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
# vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

# fig = go.Figure(data=vis_data)
# fig.show()

In [300]:
# l2r_rotmat = axangle2mat(axis=np.array([0, 1, 0]), angle=np.pi)
# r2l_rotmat = np.linalg.inv(l2r_rotmat)

# l2r_rot6d = l2r_rotmat.T.reshape(-1)[:6]
# print(l2r_rot6d.shape)

# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)


# samp = sample_grasp_q.clone()
# # samp[0, :3] = 0
# # samp[0, 3] = 1
# # samp[0, 7] = 1


# # left_rot6d = l2r_rot6d
# left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# spleft = q_left.clone()
# # spleft[0, :3] = 0
# spleft[0, 3:9] = torch.tensor(left_rot6d).to(device)



# vis_data = source_model.get_plotly_data(q=samp, color='red', opacity=0.2)
# vis_data += sm_left.get_plotly_data(q=spleft, color='blue', opacity=0.1)

# fig = go.Figure(data=vis_data)
# fig.show()

In [301]:
# rotmat_flip = rotation_matrix_from_vectors(r_palm_normal, r_palm_normal_flip)
# rotmat_flip

In [302]:
hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# hand_ply = f"{frame_id}_{1}.ply"

mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
print(mano_mesh)

x, y, z = mano_mesh.vertices.T
# i, j, k = mano_mesh.faces.T

vis_data = []
vis_data += source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.1)


# new_g = sample_grasp_q.clone()
# new_rot6d = (rotmat_flip @ palm_basis).T.reshape(-1)[:6]
# new_g[0, 3:9] = torch.tensor(new_rot6d)
# vis_data += source_model.get_plotly_data(q=new_g, color='blue', opacity=0.1)

vis_data += target_gripper_mesh_data

# left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# q_left = sample_grasp_q.clone()
# q_left[0, 9:] = -1 * torch.tensor(angles_left)
# q_left[0, 3:9] = torch.tensor(left_rot6d).to(device)
# vis_data += sm_left.get_plotly_data(q=q_left, color='blue', opacity=0.1)

vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]


fig = go.Figure(data=vis_data)
fig.show()
fig.write_html(f"../data/gtransfer_{frame_id}_{int(not use_left_hand)}.html")


<trimesh.PointCloud(vertices.shape=(778, 3), name=`000033_0.ply`)>


## Viz Mano + URDF

In [303]:
# hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# # hand_ply = f"{frame_id}_{1}.ply"

# mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
# print(mano_mesh)

# x, y, z = mano_mesh.vertices.T
# # i, j, k = mano_mesh.faces.T

# vis_data = []
# vis_data += source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
# # vis_data += target_gripper_mesh_data

# # left_rot6d = (palm_basis @ l2r_rotmat).T.reshape(-1)[:6]
# # q_left = sample_grasp_q.clone()
# # q_left[0, 9:] = -1 * torch.tensor(angles_left)
# # q_left[0, 3:9] = torch.tensor(left_rot6d).to(device)
# # vis_data += sm_left.get_plotly_data(q=q_left, color='blue', opacity=0.1)

# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]


# fig = go.Figure(data=vis_data)
# fig.show()


In [304]:
verts = np.array(mano_mesh.vertices)
center = np.mean(verts, axis=0)
print(center.shape)

all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# all_verts += mano_trans

new_grasp = sample_grasp_q.clone()
new_grasp[0, :3] -= torch.tensor(mano_trans)
vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)

x,y,z = all_verts.T
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='blue')
        )
    ]

# mano_verts = mano_mesh.vertices
# mano_verts = verts - mano_trans
# x,y,z = mano_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



fig = go.Figure(data=vis_data)
fig.show()




(3,)


In [305]:
# mano_trans

In [306]:
# print(data['right'])
# print(data['opt_translation'][0])
# print(center)


In [307]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)




# cv = verts - center
# cv[:, 0] *= -1
# # cv[:, 1] *= -1
# nv = cv + center
# x, y, z = nv.T


# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# all_verts += mano_trans

# x, y, z = all_verts.T
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)


# flip_grasp = sample_grasp_q.clone()
# flip_grasp[0, :3] -= torch.tensor(mano_trans)
# flip_grasp[0, 0] *= -1
# flip_grasp[0, :3] += torch.tensor(mano_trans)

# vis_data = source_model.get_plotly_data(q=flip_grasp, color='orange', opacity=0.5)

# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# x,y,z = mano_mesh.vertices.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")




In [308]:
# print(mano_trans)

In [309]:
# verts = np.array(mano_mesh.vertices)
# center = np.mean(verts, axis=0)
# print(center.shape)

# all_verts = verts - mano_trans
# all_verts[:, 0] *= -1
# # all_verts += mano_trans



# new_grasp = sample_grasp_q.clone()
# new_grasp[0, :3] -= torch.tensor(mano_trans)

# gg_grasp = new_grasp.clone()
# gg_grasp[0, 0] *= -1
# # vis_data = []
# vis_data = source_model.get_plotly_data(q=new_grasp, color='red', opacity=0.2)


# vis_data += source_model.get_plotly_data(q=gg_grasp, color='orange', opacity=0.2)

# x,y,z = all_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='blue')
#         )
#     ]

# mano_verts = mano_mesh.vertices
# mano_verts = verts - mano_trans
# x,y,z = mano_verts.T
# vis_data += [
#         go.Scatter3d(
#             x=x, y=y, z=z,
#             mode='markers',
#             marker=dict(size=2, color='green')
#         )
#     ]



# fig = go.Figure(data=vis_data)
# fig.show()
# # fig.write_html("gtransfer_test.html")


